# Imports + load uplift predictions

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pulp

UPLIFT_PATH = Path(
    "results/promotion_uplift_tlearner_v1.parquet"
)

HISTORY_PATH = Path(
    "data/synthetic/gta_store_product_week_history_v1.parquet"
)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

uplift = pd.read_parquet(UPLIFT_PATH)
history = pd.read_parquet(HISTORY_PATH)

uplift["week_start"] = pd.to_datetime(
    uplift["week_start"]
)

history["week_start"] = pd.to_datetime(
    history["week_start"]
)

print("Uplift:", uplift.shape)
print("History:", history.shape)

Uplift: (54000, 15)
History: (468000, 54)


# Add economic information

In [2]:
economic_cols = [
    "week_start",
    "store_id",
    "product_id",

    "margin_rate",
    "vendor_funding_rate",
    "inventory",

    # ONLY for evaluation
    "baseline_expected_units",
    "true_uplift_rate"
]

econ = (
    history[economic_cols]
    .drop_duplicates(
        subset=[
            "week_start",
            "store_id",
            "product_id"
        ]
    )
)

df = uplift.merge(
    econ,
    on=[
        "week_start",
        "store_id",
        "product_id"
    ],
    how="left",
    validate="one_to_one"
)

print(df.shape)
print(df.isna().sum().sort_values(ascending=False).head())

(54000, 20)
week_start                 0
store_id                   0
baseline_expected_units    0
inventory                  0
vendor_funding_rate        0
dtype: int64


# Calculate promotion economics

In [3]:
df["unit_cost"] = (
    df["regular_price"]
    * (
        1 - df["margin_rate"]
    )
)

In [4]:
df["promo_price"] = (
    df["regular_price"]
    * (
        1
        - df["standard_discount_pct"] / 100
    )
)

In [5]:
df["vendor_funding_per_unit"] = (
    df["regular_price"]
    * (
        df["standard_discount_pct"] / 100
    )
    * df["vendor_funding_rate"]
)

In [6]:
df["treated_unit_cost"] = (
    df["unit_cost"]
    - df["vendor_funding_per_unit"]
)

In [7]:
df["control_unit_margin"] = (
    df["regular_price"]
    - df["unit_cost"]
)

df["treated_unit_margin"] = (
    df["promo_price"]
    - df["treated_unit_cost"]
)

# Cap predicted sales by inventory

In [8]:
df["pred_control_units"] = np.minimum(
    df["pred_units_control"].clip(lower=0),
    df["inventory"]
)

df["pred_treated_units"] = np.minimum(
    df["pred_units_treated"].clip(lower=0),
    df["inventory"]
)

# Predicted profit

In [9]:
df["pred_profit_control"] = (
    df["pred_control_units"]
    * df["control_unit_margin"]
)

In [10]:
df["pred_profit_treated"] = (
    df["pred_treated_units"]
    * df["treated_unit_margin"]
)

In [11]:
df["pred_incremental_profit"] = (
    df["pred_profit_treated"]
    - df["pred_profit_control"]
)

# Promotion budget cost

In [12]:
df["retailer_discount_cost_per_unit"] = (
    df["regular_price"]
    * (
        df["standard_discount_pct"] / 100
    )
    * (
        1 - df["vendor_funding_rate"]
    )
)

In [13]:
df["promo_spend_proxy"] = (
    df["pred_treated_units"]
    * df["retailer_discount_cost_per_unit"]
)

# Create proper hidden ground truth

In [14]:
df["true_control_units_eval"] = np.minimum(
    df["baseline_expected_units"],
    df["inventory"]
)

df["true_treated_units_eval"] = np.minimum(
    df["baseline_expected_units"]
    * (
        1 + df["true_uplift_rate"]
    ),
    df["inventory"]
)

In [15]:
df["true_profit_control_eval"] = (
    df["true_control_units_eval"]
    * df["control_unit_margin"]
)

df["true_profit_treated_eval"] = (
    df["true_treated_units_eval"]
    * df["treated_unit_margin"]
)

df["true_incremental_profit_eval"] = (
    df["true_profit_treated_eval"]
    - df["true_profit_control_eval"]
)

# Inspect predictions

In [16]:
print(
    "Correlation predicted vs true profit:",
    round(
        df[
            [
                "pred_incremental_profit",
                "true_incremental_profit_eval"
            ]
        ]
        .corr()
        .iloc[0, 1],
        3
    )
)

display(
    df[
        [
            "store_id",
            "product_name",
            "category",
            "predicted_uplift_units",
            "pred_incremental_profit",
            "true_incremental_profit_eval"
        ]
    ]
    .sort_values(
        "pred_incremental_profit",
        ascending=False
    )
    .head(20)
)

Correlation predicted vs true profit: 0.921


,store_id,product_name,category,predicted_uplift_units,pred_incremental_profit,true_incremental_profit_eval
19343,1150,Snacks Product 22,Snacks,14.089789,35.653683,11.961530
39873,3106,Hot Beverages Product 23,Hot Beverages,6.981329,33.426641,15.329229
42709,3106,Ready-to-Eat Product 20,Ready-to-Eat,9.395263,30.952110,8.694835
39874,3106,Hot Beverages Product 23,Hot Beverages,6.339791,29.664278,15.329229
42485,3106,Ready-to-Eat Product 01,Ready-to-Eat,11.435763,26.618748,13.180219
39871,3106,Hot Beverages Product 23,Hot Beverages,5.647343,25.323237,15.329229
19341,1150,Snacks Product 22,Snacks,10.556524,23.883570,11.961530
40938,3106,Snacks Product 22,Snacks,10.840572,23.585494,12.767899
39650,3106,Hot Beverages Product 05,Hot Beverages,7.397168,22.994545,13.911436
41145,3106,Household Staples Product 09,Household Staples,9.673050,22.809779,3.088933


# Actual Optimization

# Install PuLP

In [17]:
%pip install pulp -q

Note: you may need to restart the kernel to use updated packages.


# Business constraints

In [18]:
MAX_FLYER_SLOTS = 20

MAX_PRODUCTS_PER_CATEGORY = 4

PROMO_BUDGET_PER_STORE_WEEK = 300

# Optimizer

In [19]:
def optimize_group(
    group,
    objective_col="pred_incremental_profit",
    max_slots=20,
    max_per_category=4,
    budget=300
):

    group = (
        group
        .reset_index(drop=True)
        .copy()
    )

    # No reason to select predicted negative-profit products
    candidates = group[
        group[objective_col] > 0
    ].copy()

    if candidates.empty:
        return pd.DataFrame()

    candidates = candidates.reset_index(drop=True)

    problem = pulp.LpProblem(
        "Retail_Promotion_Optimization",
        pulp.LpMaximize
    )

    x = {
        i: pulp.LpVariable(
            f"x_{i}",
            cat="Binary"
        )
        for i in candidates.index
    }

    # Objective
    problem += pulp.lpSum(
        candidates.loc[i, objective_col]
        * x[i]
        for i in candidates.index
    )

    # Flyer slots
    problem += (
        pulp.lpSum(
            x[i]
            for i in candidates.index
        )
        <= max_slots
    )

    # Promotion budget
    problem += (
        pulp.lpSum(
            candidates.loc[
                i,
                "promo_spend_proxy"
            ]
            * x[i]
            for i in candidates.index
        )
        <= budget
    )

    # Category diversity
    for category in (
        candidates["category"]
        .dropna()
        .unique()
    ):

        idx = candidates.index[
            candidates["category"]
            == category
        ]

        problem += (
            pulp.lpSum(
                x[i]
                for i in idx
            )
            <= max_per_category
        )

    solver = pulp.PULP_CBC_CMD(
        msg=False
    )

    problem.solve(solver)

    selected_indices = [
        i for i in candidates.index
        if x[i].value() == 1
    ]

    selected = (
        candidates.loc[
            selected_indices
        ]
        .copy()
    )

    if selected.empty:
        return selected

    selected = (
        selected
        .sort_values(
            objective_col,
            ascending=False
        )
        .reset_index(drop=True)
    )

    selected["optimized_rank"] = (
        np.arange(
            1,
            len(selected) + 1
        )
    )

    return selected

# Optimize every store/week

In [20]:
optimized_frames = []

for (
    week_start,
    store_id
), group in df.groupby(
    [
        "week_start",
        "store_id"
    ]
):

    selected = optimize_group(
        group,
        objective_col="pred_incremental_profit",
        max_slots=MAX_FLYER_SLOTS,
        max_per_category=MAX_PRODUCTS_PER_CATEGORY,
        budget=PROMO_BUDGET_PER_STORE_WEEK
    )

    if not selected.empty:

        selected["selection_method"] = (
            "Profit Optimizer"
        )

        optimized_frames.append(
            selected
        )

optimized = pd.concat(
    optimized_frames,
    ignore_index=True
)

print("Selected rows:", len(optimized))

print(
    "Store-weeks:",
    optimized[
        [
            "week_start",
            "store_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

Selected rows: 3600
Store-weeks: 180


# QA constraints

In [21]:
qa = (
    optimized
    .groupby(
        [
            "week_start",
            "store_id"
        ]
    )
    .agg(
        flyer_slots=(
            "product_id",
            "count"
        ),

        promo_spend=(
            "promo_spend_proxy",
            "sum"
        )
    )
    .reset_index()
)

print(
    "Max flyer slots:",
    qa["flyer_slots"].max()
)

print(
    "Max budget used:",
    round(
        qa["promo_spend"].max(),
        2
    )
)

Max flyer slots: 20
Max budget used: 298.02


In [22]:
category_qa = (
    optimized
    .groupby(
        [
            "week_start",
            "store_id",
            "category"
        ]
    )
    .size()
)

print(
    "Maximum category count:",
    category_qa.max()
)

Maximum category count: 4


# Compare against naive strategies

# Greedy baseline helper

In [23]:
def greedy_select(
    group,
    score_col,
    max_slots=20,
    max_per_category=4,
    budget=300
):

    group = (
        group
        .sort_values(
            score_col,
            ascending=False
        )
        .copy()
    )

    selected = []

    category_counts = {}

    total_budget = 0

    for _, row in group.iterrows():

        category = row["category"]

        if (
            category_counts.get(
                category,
                0
            )
            >= max_per_category
        ):
            continue

        new_budget = (
            total_budget
            + row["promo_spend_proxy"]
        )

        if new_budget > budget:
            continue

        selected.append(
            row.to_dict()
        )

        category_counts[category] = (
            category_counts.get(
                category,
                0
            )
            + 1
        )

        total_budget = new_budget

        if len(selected) >= max_slots:
            break

    return pd.DataFrame(selected)

# Largest-discount baseline

In [24]:
discount_frames = []

for keys, group in df.groupby(
    [
        "week_start",
        "store_id"
    ]
):

    selected = greedy_select(
        group,
        "standard_discount_pct",
        MAX_FLYER_SLOTS,
        MAX_PRODUCTS_PER_CATEGORY,
        PROMO_BUDGET_PER_STORE_WEEK
    )

    if not selected.empty:

        selected["selection_method"] = (
            "Largest Discount"
        )

        discount_frames.append(
            selected
        )

largest_discount = pd.concat(
    discount_frames,
    ignore_index=True
)

# Highest-uplift baseline

In [25]:
uplift_frames = []

for keys, group in df.groupby(
    [
        "week_start",
        "store_id"
    ]
):

    selected = greedy_select(
        group,
        "predicted_uplift_units",
        MAX_FLYER_SLOTS,
        MAX_PRODUCTS_PER_CATEGORY,
        PROMO_BUDGET_PER_STORE_WEEK
    )

    if not selected.empty:

        selected["selection_method"] = (
            "Highest Predicted Uplift"
        )

        uplift_frames.append(
            selected
        )

highest_uplift = pd.concat(
    uplift_frames,
    ignore_index=True
)

# Oracle benchmark

In [26]:
oracle_frames = []

for keys, group in df.groupby(
    [
        "week_start",
        "store_id"
    ]
):

    selected = optimize_group(
        group,
        objective_col="true_incremental_profit_eval",
        max_slots=MAX_FLYER_SLOTS,
        max_per_category=MAX_PRODUCTS_PER_CATEGORY,
        budget=PROMO_BUDGET_PER_STORE_WEEK
    )

    if not selected.empty:

        selected["selection_method"] = (
            "Oracle"
        )

        oracle_frames.append(
            selected
        )

oracle = pd.concat(
    oracle_frames,
    ignore_index=True
)

# Evaluate business performance

In [27]:
def summarize_strategy(
    data,
    name
):

    return {
        "strategy": name,

        "selected_products":
            len(data),

        "predicted_incremental_profit":
            data[
                "pred_incremental_profit"
            ].sum(),

        "true_incremental_profit":
            data[
                "true_incremental_profit_eval"
            ].sum(),

        "avg_true_profit_per_selection":
            data[
                "true_incremental_profit_eval"
            ].mean(),

        "positive_profit_rate_pct":
            (
                data[
                    "true_incremental_profit_eval"
                ] > 0
            ).mean() * 100
    }

# Compare strategies

In [28]:
comparison = pd.DataFrame([
    summarize_strategy(
        largest_discount,
        "Largest Discount"
    ),

    summarize_strategy(
        highest_uplift,
        "Highest Predicted Uplift"
    ),

    summarize_strategy(
        optimized,
        "Profit Optimizer"
    ),

    summarize_strategy(
        oracle,
        "Oracle"
    )
])

display(
    comparison.sort_values(
        "true_incremental_profit",
        ascending=False
    )
)

,strategy,selected_products,predicted_incremental_profit,true_incremental_profit,avg_true_profit_per_selection,positive_profit_rate_pct
3,Oracle,3600,15968.325853,26451.582075,7.347662,100.000000
2,Profit Optimizer,3600,22296.018504,20615.427117,5.726508,97.055556
1,Highest Predicted Uplift,3242,-10961.227337,-10097.664296,-3.114640,38.741518
0,Largest Discount,3312,-21846.524949,-16619.311787,-5.017908,26.751208


# Oracle regret

In [29]:
optimizer_profit = (
    optimized[
        "true_incremental_profit_eval"
    ].sum()
)

oracle_profit = (
    oracle[
        "true_incremental_profit_eval"
    ].sum()
)

regret_pct = (
    (
        oracle_profit
        - optimizer_profit
    )
    / oracle_profit
    * 100
)

print(
    "Optimizer regret vs Oracle:",
    round(
        regret_pct,
        2
    ),
    "%"
)

Optimizer regret vs Oracle: 22.06 %


# Improvement over discount strategy

In [30]:
discount_profit = (
    largest_discount[
        "true_incremental_profit_eval"
    ].sum()
)

improvement = (
    (
        optimizer_profit
        - discount_profit
    )
    / abs(discount_profit)
    * 100
)

print(
    "Profit improvement vs "
    "largest discount:",
    round(
        improvement,
        2
    ),
    "%"
)

Profit improvement vs largest discount: 224.05 %


# Example store flyer

In [31]:
example_store = (
    optimized["store_id"]
    .iloc[0]
)

example_week = (
    optimized[
        optimized["store_id"]
        == example_store
    ]["week_start"]
    .iloc[0]
)

example = optimized[
    (optimized["store_id"] == example_store)
    &
    (
        optimized["week_start"]
        == example_week
    )
].copy()

display(
    example[
        [
            "optimized_rank",
            "store_name",
            "week_start",
            "product_name",
            "category",

            "standard_discount_pct",

            "predicted_uplift_units",

            "pred_incremental_profit",

            "true_incremental_profit_eval"
        ]
    ]
)

,optimized_rank,store_name,week_start,product_name,category,standard_discount_pct,predicted_uplift_units,pred_incremental_profit,true_incremental_profit_eval
0,1,Toronto Stockyards,2026-06-22,Hot Beverages Product 05,Hot Beverages,20,5.067445,14.073202,12.335093
1,2,Toronto Stockyards,2026-06-22,Ready-to-Eat Product 01,Ready-to-Eat,10,4.939884,10.019110,9.693836
2,3,Toronto Stockyards,2026-06-22,Hot Beverages Product 23,Hot Beverages,15,2.707514,9.030348,13.592232
3,4,Toronto Stockyards,2026-06-22,Ready-to-Eat Product 14,Ready-to-Eat,15,5.032420,7.108750,8.982937
4,5,Toronto Stockyards,2026-06-22,Snacks Product 20,Snacks,15,4.571715,6.244374,3.908958
5,6,Toronto Stockyards,2026-06-22,Ready-to-Eat Product 04,Ready-to-Eat,10,2.867721,5.873993,4.379857
6,7,Toronto Stockyards,2026-06-22,Hot Beverages Product 27,Hot Beverages,5,2.349996,5.854838,2.207620
7,8,Toronto Stockyards,2026-06-22,Frozen Food Product 30,Frozen Food,15,1.786116,5.637559,4.976937
8,9,Toronto Stockyards,2026-06-22,Ready-to-Eat Product 27,Ready-to-Eat,5,2.931715,4.967725,5.027808
9,10,Toronto Stockyards,2026-06-22,Snacks Product 28,Snacks,10,2.788839,4.857954,3.741532


# Save results

In [32]:
OPTIMIZED_PATH = (
    RESULTS_DIR /
    "profit_optimized_promotions_v2.parquet"
)

optimized.to_parquet(
    OPTIMIZED_PATH,
    index=False
)

COMPARISON_PATH = (
    RESULTS_DIR /
    "promotion_strategy_comparison_v2.csv"
)

comparison.to_csv(
    COMPARISON_PATH,
    index=False
)

print("Saved:", OPTIMIZED_PATH)
print("Saved:", COMPARISON_PATH)

Saved: results/profit_optimized_promotions_v2.parquet
Saved: results/promotion_strategy_comparison_v2.csv
